# 03 - PV hosting capacity

## Objective

Understand hosting capacity as a search result tied to a declared criterion, then compare a small direct OpenDSS bracket with the CEPT public CLI result.

## Source, assumptions, and units

The source is the IEEE13 feeder bundled in the installed CEPT wheel. The demonstrator adds a three-phase, unity-power-factor PV at bus `675`. The overvoltage criterion is `v_max = 1.05 pu`; candidate PV sizes are in kW. These values are declared teaching inputs, not an interconnection study or a universal hosting-capacity limit.

## Prediction

Increasing PV active power should increase the feeder's maximum unregulated voltage in this setup. The 1000 kW and 2000 kW direct points should bracket the criterion, and the CEPT result should report a bus-specific value in that bracket.

## Action

Run the direct OpenDSS bracket, then stream `cept study demo hosting-capacity` into this notebook. The CLI writes an exact run directory instead of leaving the result only in memory.

## Verification

Read the actual baseline and hosting-capacity rows from `results.json`, run `cept study verify`, and assert the declared bracket and criterion.

## Interpretation

The returned `hc_kw` is criterion-specific and model-specific. It is not utility approval, a thermal rating, a protection result, or a project-validation claim.

## Exercise

Change `DIRECT_SIZES_KW` or the displayed criterion only after stating a new prediction. Rerun from a restarted kernel and keep the exact input and run path with the resulting table.

## Runtime requirements

Use Python 3.10 or newer with an existing installed `cept` command, or provide a caller-owned wheel through `CEPT_WHEEL_URL` and its exact `CEPT_WHEEL_SHA256`. The wheel must provide CEPT, OpenDSSDirect.py, and the bundled IEEE13 source files. No released PyPI version is assumed. Jupyter is needed only to execute the notebook.

In [1]:
#@title Setup — run once, then read the results below
import contextlib, io, urllib.request, hashlib
_HELPER_URL = "https://raw.githubusercontent.com/sarutesri/cept-studio-edu/75b4f394aa096a604123e6e1739d2581e8ef9476/public/notebooks/_lesson.py"
_HELPER_SHA256 = "8618face0c62b85127ffadc7b662bc177ab3ba5a36e32a09ee5223f562e02ba2"
_blob = urllib.request.urlopen(_HELPER_URL, timeout=60).read()
assert hashlib.sha256(_blob).hexdigest() == _HELPER_SHA256, "lesson helper hash mismatch"
_helper_output = io.StringIO()
with contextlib.redirect_stdout(_helper_output):
    exec(compile(_blob, "lesson helper", "exec"))
for _line in _helper_output.getvalue().splitlines():
    if "lesson helpers ready" not in _line.lower():
        print(_line)
print("Lesson helpers ready. Stage cells below run the same CEPT commands as a normal terminal.")


CEPT_WHEEL_URL not supplied; using the existing installed environment.
CLI: cept --version
cept-power-studio 0.2.0.dev0
Lesson helpers ready. Stage cells below run the same CEPT commands as a normal terminal.


### Direct sweep - same network, larger PV

Each step rebuilds the base case, adds one PV plant, and solves. The per-step asserts stop a non-converged step immediately.


In [2]:
#@title Under the hood - direct OpenDSS sweep (optional)
MASTER_DSS = ieee13_master()
DIRECT_SIZES_KW = (0, 1000, 2000)
V_MAX_PU = 1.05
import opendssdirect as dss

def load_base():
    dss.Basic.ClearAll()
    dss.Basic.DataPath(str(MASTER_DSS.parent))
    dss.Text.Command(f'Redirect "{MASTER_DSS}"')
    dss.Text.Command('CalcVoltageBases')
    dss.Text.Command('Solve')
    assert dss.Solution.Converged()

def max_unregulated_voltage():
    excluded = {'sourcebus', '650', 'rg60'}
    names = dss.Circuit.AllNodeNames()
    values = dss.Circuit.AllBusMagPu()
    return max(value for name, value in zip(names, values) if name.split('.')[0].lower() not in excluded)

direct_sweep = {}
for kw in DIRECT_SIZES_KW:
    load_base()
    dss.Text.Command(f'New PVSystem.lesson_pv phases=3 bus1=675.1.2.3 kV=4.16 kVA={max(kw, 1)} Pmpp={kw} irradiance=1 pf=1 %cutin=0.05 %cutout=0.05')
    dss.Text.Command('Solve')
    assert dss.Solution.Converged()
    direct_sweep[kw] = max_unregulated_voltage()
os.chdir(WORKSPACE)
print(f"Direct OpenDSS sweep finished: solved {len(direct_sweep)} PV sizes.")


Direct OpenDSS sweep finished: solved 3 PV sizes.


### Read the sweep back

Maximum unregulated voltage per PV size. The bracket assert is the lesson: 1000 kW holds, 2000 kW breaks the 1.05 pu ceiling.


In [3]:
#@title Under the hood — direct sweep readback (optional)
table(['PV size', 'maximum unregulated voltage', 'unit'], [(kw, value, 'pu') for kw, value in direct_sweep.items()])
assert direct_sweep[1000] < V_MAX_PU < direct_sweep[2000]

| PV size | maximum unregulated voltage | unit |
| --- | --- | --- |
| 0 | 1.0426313303211827 | pu |
| 1000 | 1.0476066810380371 | pu |
| 2000 | 1.0521988003106502 | pu |


### Run the search through CEPT

The main CEPT path is the terminal command below. The direct three-point sweep above remains an optional reference bracket, not a second product workflow.

In [4]:
!cept study demo hosting-capacity \
    --network ieee13 \
    --out runs/03-hosting-capacity \
    --force \
    --format text

!cept study verify runs/03-hosting-capacity --format text

CEPT study result: FINISHED
----------------------------
Result             Finished the hosting-capacity search and saved the evidence
Saved run          runs\03-hosting-capacity
Case fingerprint   849d2148e0b1 (matches the case you ran)

What this means
  The study completed and saved solver-backed evidence.
  It does NOT approve a real project or field installation.

Next
  cept study verify runs\03-hosting-capacity --format text


CEPT study check: PASSED
----------------------------
Study              Hosting capacity (OpenDSS)
Case fingerprint   849d2148e0b1 (matches the case you ran)

Checked   3 groups, 13 checks, all passed
  [PASS] Case identity (4 checks)
  [PASS] Solver result (2 checks)
  [PASS] Saved evidence (7 checks)

What this means
  The saved result matches its Case, solver run, and saved evidence.
  It does NOT approve a real project or field installation.

Saved evidence     runs\03-hosting-capacity\public-verification.json

For the full check list
  cept study verify runs\03-hosting-capacity --format json


### Compare search with the direct bracket

The optional detail cell reads the persisted hosting-capacity result and checks that bus 675 lands inside the direct 1000–2000 kW bracket under the declared 1.05 pu overvoltage criterion.

In [5]:
#@title Compare solver outputs (optional details)
RUN_DIR = WORKSPACE / "runs" / "03-hosting-capacity"
results = read(RUN_DIR / "results.json")
verify_summary = read(RUN_DIR / "public-verification.json")
hosting = results["hosting_capacity"]
item = next(row for row in hosting["items"] if row["bus"].lower() == "675")
table(
    ["field", "value", "unit"],
    [
        ("criterion", hosting["criterion"], "text"),
        ("v_max", hosting["v_max_pu"], "pu"),
        ("baseline_v_max", hosting["baseline_v_max_pu"], "pu"),
        ("bus 675 capacity", item["hc_kw"], "kW"),
        ("bus 675 limit", item["limit"], "text"),
    ],
)
print()
print("CEPT hosting-capacity result")
print("----------------------------")
print(f"Result        {'PASSED' if verify_summary['passed'] else 'Needs attention'}")
print(f"Bus 675 can host about {item['hc_kw']} kW before hitting the voltage limit")
print(f"Limit reached: {item['limit']}")
assert verify_summary["status"] == "PASS"
assert verify_summary["passed"] is True
assert abs(direct_sweep[0] - hosting["baseline_v_max_pu"]) < 1e-3
assert 1000 <= item["hc_kw"] <= 2000
assert hosting["criterion"] == "overvoltage" and hosting["v_max_pu"] == V_MAX_PU


| field | value | unit |
| --- | --- | --- |
| criterion | overvoltage | text |
| v_max | 1.05 | pu |
| baseline_v_max | 1.0426 | pu |
| bus 675 capacity | 1507.6 | kW |
| bus 675 limit | overvoltage | text |

CEPT hosting-capacity result
----------------------------
Result        PASSED
Bus 675 can host about 1507.6 kW before hitting the voltage limit
Limit reached: overvoltage


The rows above are read from solver-backed direct OpenDSS values and the exact CEPT `results.json`. The bracket supports a teaching interpretation of this demonstrator only. A real hosting-capacity decision needs source-bound ratings, scenarios, protection and control settings, applicable criteria, and reviewer acceptance.